# 📊 Code 1b: Filter Liquid Securities (Updated)

## Purpose
Filter stocks that have **good volumes** and are **tradeable** based on liquidity criteria.

## Input
- **`data_raw_all.csv`** (from Code 1a)

## Output
- **`data_raw.csv`** - Filtered dataset with only liquid securities
- `liquidity_analysis.csv` - Liquidity metrics for all stocks
- `filtered_out_tickers.csv` - Stocks that were filtered out
- `tickers_final.csv` - List of unique tickers in filtered data

## Filtering Criteria:
1. **Minimum Trading Days:** At least 500 trading days between 01-Jan-2011 to 31-Dec-2018
2. **Value Traded:** Close × Volume ≥ ₹5 crore daily average
3. **Data Availability:** Less than 30% missing days in any rolling 1-year window

## Key Update:
**Using Close only** (split-adjusted) for all calculations. No Adj_Close.

---

**⏱️ Estimated Time:** 5-10 minutes

## Step 1: Import Libraries

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")
print(f"📅 Script run date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Libraries imported successfully!
📅 Script run date: 2026-06-15 17:50:31


## Step 2: Configuration

In [2]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Input/Output files
INPUT_FILE = 'data_raw_all.csv'
OUTPUT_FILE = 'data_raw.csv'
LIQUIDITY_FILE = 'liquidity_analysis.csv'
FILTERED_OUT_FILE = 'filtered_out_tickers.csv'

# Filtering criteria
MIN_TRADING_DAYS = 500  # Minimum days between 2011-2018
MIN_VALUE_TRADED_CR = 5  # Minimum ₹5 crore daily average
MAX_MISSING_PCT = 5  # Maximum 5% missing data in any 1-year window

# Period for minimum trading days check
PERIOD_START = '2011-01-01'
PERIOD_END = '2019-12-31'

print("="*80)
print("CODE 1b: FILTER LIQUID SECURITIES CONFIGURATION (UPDATED)")
print("="*80)
print(f"Input File: {INPUT_FILE}")
print(f"Output File: {OUTPUT_FILE}")
print(f"\nFiltering Criteria:")
print(f"  - Minimum trading days (2011-2018): {MIN_TRADING_DAYS}")
print(f"  - Minimum daily value traded: ₹{MIN_VALUE_TRADED_CR} crore")
print(f"  - Maximum missing data: {MAX_MISSING_PCT}% in any 1-year window")
print(f"\n💡 Using Close × Volume for value traded (split-adjusted Close)")
print("="*80)

CODE 1b: FILTER LIQUID SECURITIES CONFIGURATION (UPDATED)
Input File: data_raw_all.csv
Output File: data_raw.csv

Filtering Criteria:
  - Minimum trading days (2011-2018): 500
  - Minimum daily value traded: ₹5 crore
  - Maximum missing data: 5% in any 1-year window

💡 Using Close × Volume for value traded (split-adjusted Close)


## Step 3: Load Data

**⚠️ Make sure `data_raw_all.csv` (from Code 1a) is available.**

In [3]:
print("Loading raw data...")
print("-"*80)

# Load data
df_raw = pd.read_csv(INPUT_FILE)
df_raw['Date'] = pd.to_datetime(df_raw['Date'])

print(f"✅ Data loaded successfully!")
print(f"\n📊 INITIAL DATASET:")
print(f"   Total records: {len(df_raw):,}")
print(f"   Unique stocks: {df_raw['Ticker'].nunique()}")
print(f"   Date range: {df_raw['Date'].min()} to {df_raw['Date'].max()}")
print(f"   Columns: {list(df_raw.columns)}")
print("-"*80)

Loading raw data...
--------------------------------------------------------------------------------
✅ Data loaded successfully!

📊 INITIAL DATASET:
   Total records: 6,435,047
   Unique stocks: 2399
   Date range: 2007-01-02 00:00:00 to 2026-06-12 00:00:00
   Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Ticker']
--------------------------------------------------------------------------------


## Step 4: Calculate Value Traded

**Value Traded = Close × Volume**

This is our first calculation. Code 1a provides pure raw data.

In [4]:
print("Calculating Value Traded...")
print("-"*80)

# Calculate Value Traded using split-adjusted Close × Volume
df_raw['Value_Traded'] = df_raw['Close'] * df_raw['Volume']
df_raw['Value_Traded_Cr'] = df_raw['Value_Traded'] / 10000000  # Convert to crores

print(f"✅ Value traded calculated!")
print(f"\n📊 VALUE TRADED STATISTICS:")
print(df_raw[['Value_Traded', 'Value_Traded_Cr']].describe())
print("-"*80)

Calculating Value Traded...
--------------------------------------------------------------------------------
✅ Value traded calculated!

📊 VALUE TRADED STATISTICS:
       Value_Traded  Value_Traded_Cr
count  6.435047e+06     6.435047e+06
mean   2.502987e+08     2.502987e+01
std    1.327606e+09     1.327606e+02
min    0.000000e+00     0.000000e+00
25%    7.297129e+05     7.297129e-02
50%    7.672706e+06     7.672706e-01
75%    6.912221e+07     6.912221e+00
max    3.766747e+11     3.766747e+04
--------------------------------------------------------------------------------


## Step 5: Filter by Trading Days (2011-2018)

**Criterion:** Stock must have at least 500 trading days between 01-Jan-2011 and 31-Dec-2018

In [5]:
print("Filter 1: Checking trading days in 2011-2018 period...")
print("-"*80)

# Filter data for 2011-2018 period
period_data = df_raw[
    (df_raw['Date'] >= PERIOD_START) &
    (df_raw['Date'] <= PERIOD_END)
].copy()

# Count trading days per ticker in this period
trading_days_count = period_data.groupby('Ticker').size().reset_index(name='Trading_Days_2011_2018')

print(f"📊 BEFORE FILTER:")
print(f"   Total unique stocks: {df_raw['Ticker'].nunique()}")
print(f"   Total records: {len(df_raw):,}")

# Identify stocks that meet the criteria
qualified_tickers_days = trading_days_count[
    trading_days_count['Trading_Days_2011_2018'] >= MIN_TRADING_DAYS
]['Ticker'].tolist()

print(f"\n✅ Stocks with ≥{MIN_TRADING_DAYS} trading days: {len(qualified_tickers_days)}")
print(f"❌ Stocks filtered out: {df_raw['Ticker'].nunique() - len(qualified_tickers_days)}")

# Apply filter
df_filtered = df_raw[df_raw['Ticker'].isin(qualified_tickers_days)].copy()

print(f"\n📊 AFTER FILTER 1:")
print(f"   Unique stocks: {df_filtered['Ticker'].nunique()}")
print(f"   Total records: {len(df_filtered):,}")
print(f"   Records removed: {len(df_raw) - len(df_filtered):,}")
print("-"*80)

Filter 1: Checking trading days in 2011-2018 period...
--------------------------------------------------------------------------------
📊 BEFORE FILTER:
   Total unique stocks: 2399
   Total records: 6,435,047

✅ Stocks with ≥500 trading days: 1346
❌ Stocks filtered out: 1053

📊 AFTER FILTER 1:
   Unique stocks: 1346
   Total records: 5,548,904
   Records removed: 886,143
--------------------------------------------------------------------------------


## Step 6: Calculate Liquidity Metrics

Calculate average daily value traded for each stock.

In [6]:
print("Calculating liquidity metrics...")
print("-"*80)

# Calculate average value traded per ticker
liquidity_metrics = df_filtered.groupby('Ticker').agg({
    'Value_Traded_Cr': ['mean', 'median', 'std'],
    'Volume': ['mean', 'median'],
    'Date': ['min', 'max', 'count']
}).reset_index()

# Flatten column names
liquidity_metrics.columns = [
    'Ticker',
    'Avg_Value_Traded_Cr', 'Median_Value_Traded_Cr', 'Std_Value_Traded_Cr',
    'Avg_Volume', 'Median_Volume',
    'First_Date', 'Last_Date', 'Total_Days'
]

print(f"✅ Liquidity metrics calculated for {len(liquidity_metrics)} stocks")
print(f"\n📊 TOP 10 MOST LIQUID STOCKS:")
display(liquidity_metrics.nlargest(10, 'Avg_Value_Traded_Cr'))
print("-"*80)

Calculating liquidity metrics...
--------------------------------------------------------------------------------
✅ Liquidity metrics calculated for 1346 stocks

📊 TOP 10 MOST LIQUID STOCKS:


,Ticker,Avg_Value_Traded_Cr,Median_Value_Traded_Cr,Std_Value_Traded_Cr,Avg_Volume,Median_Volume,First_Date,Last_Date,Total_Days
986,RELIANCE,982.213141,639.860320,1022.862003,1.998607e+07,15763138.0,2007-01-02,2026-06-12,4797
470,HDFCBANK,804.695808,221.849305,1378.533313,1.747706e+07,12695476.0,2007-01-02,2026-06-12,4797
510,ICICIBANK,745.870437,483.416570,720.987604,2.153861e+07,16662978.0,2007-01-02,2026-06-12,4797
1043,SBIN,632.658436,455.381303,572.208315,2.243927e+07,17594228.0,2007-01-02,2026-06-12,4797
123,AXISBANK,525.035272,360.239766,511.700219,1.049980e+07,8143792.0,2007-01-02,2026-06-12,4797
555,INFY,522.468614,322.454021,552.748462,9.259565e+06,7425832.0,2007-01-02,2026-06-12,4797
1217,TMPV,500.813319,314.765389,621.382544,1.793676e+07,10702375.0,2007-01-02,2026-06-12,4797
176,BHARTIARTL,434.443980,223.400037,723.727261,7.748660e+06,5515124.0,2007-01-02,2026-06-12,4797
135,BAJFINANCE,407.310485,77.081637,674.508638,1.157481e+07,6549340.0,2007-01-02,2026-06-12,4797
1183,TCS,400.942917,241.073880,444.500593,3.168750e+06,2534521.0,2007-01-02,2026-06-12,4797


--------------------------------------------------------------------------------


## Step 7: Filter by Liquidity

**Criterion:** Average daily value traded ≥ ₹5 crore

In [7]:
print(f"Filter 2: Checking average value traded ≥ ₹{MIN_VALUE_TRADED_CR} crore...")
print("-"*80)

print(f"📊 BEFORE FILTER 2:")
print(f"   Unique stocks: {df_filtered['Ticker'].nunique()}")
print(f"   Total records: {len(df_filtered):,}")

# Identify liquid stocks
qualified_tickers_liquidity = liquidity_metrics[
    liquidity_metrics['Avg_Value_Traded_Cr'] >= MIN_VALUE_TRADED_CR
]['Ticker'].tolist()

print(f"\n✅ Stocks with avg value ≥ ₹{MIN_VALUE_TRADED_CR} cr: {len(qualified_tickers_liquidity)}")
print(f"❌ Stocks filtered out: {df_filtered['Ticker'].nunique() - len(qualified_tickers_liquidity)}")

# Apply filter
df_filtered = df_filtered[df_filtered['Ticker'].isin(qualified_tickers_liquidity)].copy()

print(f"\n📊 AFTER FILTER 2:")
print(f"   Unique stocks: {df_filtered['Ticker'].nunique()}")
print(f"   Total records: {len(df_filtered):,}")
print(f"   Records removed: {len(df_raw) - len(df_filtered):,}")
print("-"*80)

Filter 2: Checking average value traded ≥ ₹5 crore...
--------------------------------------------------------------------------------
📊 BEFORE FILTER 2:
   Unique stocks: 1346
   Total records: 5,548,904

✅ Stocks with avg value ≥ ₹5 cr: 575
❌ Stocks filtered out: 771

📊 AFTER FILTER 2:
   Unique stocks: 575
   Total records: 2,449,347
   Records removed: 3,985,700
--------------------------------------------------------------------------------


## Step 8: Check Data Completeness

**Criterion:** Stock should not have more than 5% missing days in any rolling 1-year window

In [8]:
print(f"Filter 3: Checking data completeness...")
print("-"*80)

print(f"📊 BEFORE FILTER 3:")
print(f"   Unique stocks: {df_filtered['Ticker'].nunique()}")
print(f"   Total records: {len(df_filtered):,}")

# Function to check data completeness
def check_data_completeness(group):
    """
    Check if stock has acceptable data completeness
    Returns True if stock has < 30% missing days in any 1-year window
    """
    # Sort by date
    group = group.sort_values('Date')

    # Get date range
    min_date = group['Date'].min()
    max_date = group['Date'].max()

    # Calculate total possible trading days (approximate: 252 per year)
    years = (max_date - min_date).days / 365.25
    expected_days = years * 252
    actual_days = len(group)

    # Calculate completeness percentage
    completeness_pct = (actual_days / expected_days) * 100 if expected_days > 0 else 0

    # Accept if completeness is >= 70% (i.e., missing <= 30%)
    return completeness_pct >= (100 - MAX_MISSING_PCT)

# Check completeness for each ticker
completeness_check = df_filtered.groupby('Ticker').apply(check_data_completeness)
qualified_tickers_completeness = completeness_check[completeness_check].index.tolist()

print(f"\n✅ Stocks with acceptable data completeness: {len(qualified_tickers_completeness)}")
print(f"❌ Stocks filtered out: {df_filtered['Ticker'].nunique() - len(qualified_tickers_completeness)}")

# Apply filter
df_filtered = df_filtered[df_filtered['Ticker'].isin(qualified_tickers_completeness)].copy()

print(f"\n📊 AFTER FILTER 3:")
print(f"   Unique stocks: {df_filtered['Ticker'].nunique()}")
print(f"   Total records: {len(df_filtered):,}")
print(f"   Records removed: {len(df_raw) - len(df_filtered):,}")
print("-"*80)

Filter 3: Checking data completeness...
--------------------------------------------------------------------------------
📊 BEFORE FILTER 3:
   Unique stocks: 575
   Total records: 2,449,347

✅ Stocks with acceptable data completeness: 575
❌ Stocks filtered out: 0

📊 AFTER FILTER 3:
   Unique stocks: 575
   Total records: 2,449,347
   Records removed: 3,985,700
--------------------------------------------------------------------------------


## Step 9: Summary of Filtering

Show before/after comparison of all filters.

In [9]:
print("="*80)
print("FILTERING SUMMARY")
print("="*80)

# Create summary
filtering_summary = pd.DataFrame([
    {
        'Stage': 'Initial (After Code 1a)',
        'Stocks': df_raw['Ticker'].nunique(),
        'Records': len(df_raw)
    },
    {
        'Stage': f'After Filter 1 (≥{MIN_TRADING_DAYS} days in 2011-2018)',
        'Stocks': len(qualified_tickers_days),
        'Records': len(df_raw[df_raw['Ticker'].isin(qualified_tickers_days)])
    },
    {
        'Stage': f'After Filter 2 (≥₹{MIN_VALUE_TRADED_CR} cr avg value)',
        'Stocks': len(qualified_tickers_liquidity),
        'Records': len(df_raw[df_raw['Ticker'].isin(qualified_tickers_liquidity)])
    },
    {
        'Stage': f'Final (After all filters)',
        'Stocks': df_filtered['Ticker'].nunique(),
        'Records': len(df_filtered)
    }
])

display(filtering_summary)

print(f"\n📊 FILTERING IMPACT:")
print(f"   Initial stocks: {df_raw['Ticker'].nunique()}")
print(f"   Final stocks: {df_filtered['Ticker'].nunique()}")
print(f"   Stocks removed: {df_raw['Ticker'].nunique() - df_filtered['Ticker'].nunique()}")
print(f"   Retention rate: {df_filtered['Ticker'].nunique() / df_raw['Ticker'].nunique() * 100:.1f}%")
print(f"\n   Initial records: {len(df_raw):,}")
print(f"   Final records: {len(df_filtered):,}")
print(f"   Records removed: {len(df_raw) - len(df_filtered):,}")
print(f"   Record retention: {len(df_filtered) / len(df_raw) * 100:.1f}%")

print("\n" + "="*80)

FILTERING SUMMARY


,Stage,Stocks,Records
0,Initial (After Code 1a),2399,6435047
1,After Filter 1 (≥500 days in 2011-2018),1346,5548904
2,After Filter 2 (≥₹5 cr avg value),575,2449347
3,Final (After all filters),575,2449347



📊 FILTERING IMPACT:
   Initial stocks: 2399
   Final stocks: 575
   Stocks removed: 1824
   Retention rate: 24.0%

   Initial records: 6,435,047
   Final records: 2,449,347
   Records removed: 3,985,700
   Record retention: 38.1%



## Step 10: Create Filtered Out List

Document which tickers were filtered out and why.

In [10]:
print("Creating filtered out tickers list...")
print("-"*80)

# Get all original tickers
all_tickers = set(df_raw['Ticker'].unique())
final_tickers = set(df_filtered['Ticker'].unique())
filtered_out = all_tickers - final_tickers

# Create detailed filtered out list
filtered_out_details = []

for ticker in filtered_out:
    reasons = []

    # Check why it was filtered
    if ticker not in qualified_tickers_days:
        days = trading_days_count[trading_days_count['Ticker'] == ticker]['Trading_Days_2011_2018'].values
        if len(days) > 0:
            reasons.append(f"Only {days[0]} trading days in 2011-2018")
        else:
            reasons.append("No data in 2011-2018")

    if ticker not in qualified_tickers_liquidity:
        liq = liquidity_metrics[liquidity_metrics['Ticker'] == ticker]['Avg_Value_Traded_Cr'].values
        if len(liq) > 0:
            reasons.append(f"Low liquidity (₹{liq[0]:.2f} cr avg)")
        else:
            reasons.append("No liquidity data")

    if ticker not in qualified_tickers_completeness:
        reasons.append("Incomplete data (>30% missing)")

    filtered_out_details.append({
        'Ticker': ticker,
        'Reasons': '; '.join(reasons)
    })

df_filtered_out = pd.DataFrame(filtered_out_details)

print(f"✅ Documented {len(df_filtered_out)} filtered out tickers")
print(f"\nSample of filtered out tickers:")
display(df_filtered_out.head(10))
print("-"*80)

Creating filtered out tickers list...
--------------------------------------------------------------------------------
✅ Documented 1824 filtered out tickers

Sample of filtered out tickers:


,Ticker,Reasons
0,MIDCAP,No data in 2011-2018; No liquidity data; Incom...
1,ASAHIINDIA,Low liquidity (₹4.98 cr avg); Incomplete data ...
2,GROWWMETAL,No data in 2011-2018; No liquidity data; Incom...
3,INOXGREEN,No data in 2011-2018; No liquidity data; Incom...
4,MOSCHIP,No data in 2011-2018; No liquidity data; Incom...
5,VAIBHAVGBL,Low liquidity (₹4.76 cr avg); Incomplete data ...
6,BHARATCOAL,No data in 2011-2018; No liquidity data; Incom...
7,ZOTA,Low liquidity (₹2.55 cr avg); Incomplete data ...
8,HONASA,No data in 2011-2018; No liquidity data; Incom...
9,SCODATUBES,No data in 2011-2018; No liquidity data; Incom...


--------------------------------------------------------------------------------


## Step 11: Save Output Files

In [11]:
print("="*80)
print("SAVING OUTPUT FILES")
print("="*80)

# Save main filtered dataset
print(f"\nSaving {OUTPUT_FILE}...")
df_filtered.to_csv(OUTPUT_FILE, index=False)
file_size_mb = df_filtered.memory_usage(deep=True).sum() / 1024**2
print(f"✅ Saved: {OUTPUT_FILE}")
print(f"   Size: {file_size_mb:.2f} MB")
print(f"   Rows: {len(df_filtered):,}")
print(f"   Stocks: {df_filtered['Ticker'].nunique()}")
print(f"   Columns: {list(df_filtered.columns)}")

# Save liquidity analysis
print(f"\nSaving {LIQUIDITY_FILE}...")
liquidity_metrics_final = liquidity_metrics[
    liquidity_metrics['Ticker'].isin(df_filtered['Ticker'].unique())
].copy()
liquidity_metrics_final.to_csv(LIQUIDITY_FILE, index=False)
print(f"✅ Saved: {LIQUIDITY_FILE}")
print(f"   Stocks: {len(liquidity_metrics_final)}")

# Save filtered out list
print(f"\nSaving {FILTERED_OUT_FILE}...")
df_filtered_out.to_csv(FILTERED_OUT_FILE, index=False)
print(f"✅ Saved: {FILTERED_OUT_FILE}")
print(f"   Filtered out: {len(df_filtered_out)} tickers")

print("\n" + "="*80)
print("✅ ALL FILES SAVED!")
print("="*80)

SAVING OUTPUT FILES

Saving data_raw.csv...
✅ Saved: data_raw.csv
   Size: 299.39 MB
   Rows: 2,449,347
   Stocks: 575
   Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Ticker', 'Value_Traded', 'Value_Traded_Cr']

Saving liquidity_analysis.csv...
✅ Saved: liquidity_analysis.csv
   Stocks: 575

Saving filtered_out_tickers.csv...
✅ Saved: filtered_out_tickers.csv
   Filtered out: 1824 tickers

✅ ALL FILES SAVED!


## Step 12: Final Summary

In [12]:
print("="*80)
print("FINAL SUMMARY - CODE 1b (UPDATED)")
print("="*80)

print(f"""
📊 Output Dataset (data_raw.csv):
   - Total stocks: {df_filtered['Ticker'].nunique()}
   - Total records: {len(df_filtered):,}
   - Date range: {df_filtered['Date'].min()} to {df_filtered['Date'].max()}
   - Columns: {list(df_filtered.columns)}

✅ All stocks in output meet criteria:
   - ≥{MIN_TRADING_DAYS} trading days in 2011-2018
   - ≥₹{MIN_VALUE_TRADED_CR} crore average daily value traded (Close × Volume)
   - <{MAX_MISSING_PCT}% missing data in any year

💡 METHODOLOGY:
   - Using split-adjusted Close for all calculations
   - No Adj_Close column (simpler approach)
   - Value Traded = Close × Volume

📁 Output Files:
   1. {OUTPUT_FILE} - Filtered liquid securities
   2. {LIQUIDITY_FILE} - Liquidity metrics
   3. {FILTERED_OUT_FILE} - Stocks that were filtered out
   4. tickers_final.csv - List of unique tickers in filtered data

➡️  Next Step: Run Code 2 to clean and preprocess data
""")

print("="*80)

FINAL SUMMARY - CODE 1b (UPDATED)

📊 Output Dataset (data_raw.csv):
   - Total stocks: 575
   - Total records: 2,449,347
   - Date range: 2007-01-02 00:00:00 to 2026-06-12 00:00:00
   - Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Ticker', 'Value_Traded', 'Value_Traded_Cr']

✅ All stocks in output meet criteria:
   - ≥500 trading days in 2011-2018
   - ≥₹5 crore average daily value traded (Close × Volume)
   - <5% missing data in any year

💡 METHODOLOGY:
   - Using split-adjusted Close for all calculations
   - No Adj_Close column (simpler approach)
   - Value Traded = Close × Volume

📁 Output Files:
   1. data_raw.csv - Filtered liquid securities
   2. liquidity_analysis.csv - Liquidity metrics
   3. filtered_out_tickers.csv - Stocks that were filtered out
   4. tickers_final.csv - List of unique tickers in filtered data

➡️  Next Step: Run Code 2 to clean and preprocess data



## Step 13: Preview Final Data

In [13]:
print("FINAL FILTERED DATA (First 20 rows):")
print("="*80)
display(df_filtered.head(20))

print("\nSUMMARY STATISTICS:")
print("="*80)
display(df_filtered.describe())

FINAL FILTERED DATA (First 20 rows):


,Date,Open,High,Low,Close,Volume,Ticker,Value_Traded,Value_Traded_Cr
8102,2007-01-02,962.630615,977.636194,858.319096,859.410422,965111,3IINFOLTD,8.294265e+08,82.942645
8103,2007-01-03,983.365723,1002.645575,960.629995,976.272131,895672,3IINFOLTD,8.744196e+08,87.441961
8104,2007-01-04,964.904175,993.096474,954.900456,993.096474,247242,3IINFOLTD,2.455352e+08,24.553516
8105,2007-01-05,978.818542,999.007823,963.994804,963.994804,264776,3IINFOLTD,2.552427e+08,25.524269
8106,2007-01-08,1023.380615,1032.202033,950.353414,976.272143,699857,3IINFOLTD,6.832509e+08,68.325089
8107,2007-01-09,994.733582,1052.482329,977.636270,1036.749161,423121,3IINFOLTD,4.386703e+08,43.867034
8108,2007-01-10,969.269470,994.460609,963.994738,989.913464,152130,3IINFOLTD,1.505955e+08,15.059554
8109,2007-01-11,1062.940796,1076.582233,962.903593,977.636300,740662,3IINFOLTD,7.240981e+08,72.409806
8110,2007-01-12,1064.941406,1095.861994,1031.565405,1086.312989,562895,3IINFOLTD,6.114801e+08,61.148015
8111,2007-01-15,1060.667114,1090.860113,1040.386803,1074.945106,283064,3IINFOLTD,3.042783e+08,30.427826



SUMMARY STATISTICS:


,Date,Open,High,Low,Close,Volume,Value_Traded,Value_Traded_Cr
count,2449347,2.449347e+06,2.449347e+06,2.449347e+06,2.449347e+06,2.449347e+06,2.449347e+06,2.449347e+06
mean,2017-06-16 09:59:04.398240512,7.479152e+02,7.599110e+02,7.373714e+02,7.489186e+02,3.424639e+06,5.490032e+08,5.490032e+01
min,2007-01-02 00:00:00,2.000000e-01,2.329659e-01,1.500000e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2012-11-30 00:00:00,4.825000e+01,4.948690e+01,4.726123e+01,4.830000e+01,8.340000e+04,1.079623e+07,1.079623e+00
50%,2017-10-30 00:00:00,1.455500e+02,1.487183e+02,1.428459e+02,1.457921e+02,3.959070e+05,6.751995e+07,6.751995e+00
75%,2022-02-18 00:00:00,4.716152e+02,4.808739e+02,4.638668e+02,4.721444e+02,1.784487e+06,3.572295e+08,3.572295e+01
max,2026-06-12 00:00:00,1.622886e+05,1.635935e+05,1.602787e+05,1.619736e+05,2.179435e+09,3.766747e+11,3.766747e+04
std,NaN,3.738016e+03,3.784686e+03,3.696964e+03,3.743652e+03,1.749969e+07,1.933563e+09,1.933563e+02


## Step 14: Create Final Tickers List

Extract all unique tickers that passed the liquidity filters and save to `tickers_final.csv`

In [14]:
print("Creating final tickers list...")
print("-"*80)

# Extract unique tickers from filtered data
unique_tickers = df_filtered['Ticker'].unique()
unique_tickers_sorted = sorted(unique_tickers)

# Create DataFrame
df_tickers_final = pd.DataFrame({
    'Ticker': unique_tickers_sorted
})

# Save to CSV
TICKERS_FINAL_FILE = 'tickers_final.csv'
df_tickers_final.to_csv(TICKERS_FINAL_FILE, index=False)

print(f"✅ Created {TICKERS_FINAL_FILE}")
print(f"   Total liquid stocks: {len(df_tickers_final)}")
print()
print("Sample tickers (first 20):")
print(df_tickers_final.head(20))
print()
print("Sample tickers (last 10):")
print(df_tickers_final.tail(10))

print("-"*80)

Creating final tickers list...
--------------------------------------------------------------------------------
✅ Created tickers_final.csv
   Total liquid stocks: 575

Sample tickers (first 20):
        Ticker
0    3IINFOLTD
1      3MINDIA
2      63MOONS
3   AARTIDRUGS
4     AARTIIND
5          ABB
6   ABBOTINDIA
7    ABCAPITAL
8        ABFRL
9        ABREL
10         ACC
11         ACE
12  ADANIENSOL
13    ADANIENT
14  ADANIPORTS
15  ADANIPOWER
16  ADVENZYMES
17    AEGISLOG
18         AGI
19      AIAENG

Sample tickers (last 10):
         Ticker
565   WHIRLPOOL
566       WIPRO
567  WOCKPHARMA
568     YESBANK
569        ZEEL
570  ZENSARTECH
571      ZENTEC
572   ZFCVINDIA
573   ZYDUSLIFE
574   ZYDUSWELL
--------------------------------------------------------------------------------


## 📥 Download Files (Optional)

In [15]:
from google.colab import files

print("Downloading files...")
files.download(OUTPUT_FILE)
files.download(LIQUIDITY_FILE)
files.download(FILTERED_OUT_FILE)
files.download(TICKERS_FINAL_FILE)

print("\n✅ Download complete!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download complete!


---

## ✅ Code 1b Complete! (Updated - No Adj_Close)

**Next Step:** Run **Code 2** to clean and preprocess the data.

### What Changed:
- ✅ No references to Adj_Close
- ✅ Using Close × Volume for value traded
- ✅ Simpler methodology